# Feature Store Creation Notebook

This notebook implements a workflow for processing stock market data and creating features for machine learning models.

Steps:
1. Load stock data 
2. Run indicators for building features
3. Sync features data to delta-lake

In [2]:
%pip install vectorbt
%pip install pykalman

  Using cached numba-0.61.2-cp312-cp312-macosx_11_0_arm64.whl.metadata (2.8 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached schedule-1.2.2-py3-none-any.whl.metadata (3.8 kB)
  Using cached llvmlite-0.44.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (4.8 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.7/109.7 kB 1.8 MB/s eta 0:00:00a 0:00:01
  Using cached pyparsing-3.2.3-py3-none-any.whl.metadata (5.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.7/527.7 kB 5.7 MB/s eta 0:00:00a 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 17.5 MB/s eta 0:00:00
Using cached numba-0.61.2-cp312-cp312-macosx_11_0_arm64.whl (2.8 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 15.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.5/315.5 kB 2.8 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 13.5 MB/s eta 0

In [43]:
import os
import sys
from pathlib import Path
import vectorbt as vbt
from pykalman import KalmanFilter
import pandas as pd
from deltalake import DeltaTable, write_deltalake
from dotenv import load_dotenv
load_dotenv()

# Set up the Python path for notebook environment
# Navigate from backend/scripts to backend (project root)
CURRENT_DIR = os.getcwd()
if 'scripts' in CURRENT_DIR:
    PROJECT_ROOT = os.path.dirname(CURRENT_DIR)  # Go up from scripts to backend
else:
    PROJECT_ROOT = CURRENT_DIR

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Current directory: {CURRENT_DIR}")
print(f"Project root: {PROJECT_ROOT}")

from app.services.indicators import zscore_nb, avwap_func_nb, relative_strength_nb, yang_zhang_volatility_nb, directional_change_nb

Current directory: /Users/phuchdh/Documents/Work/all-in-one-portfolio/backend/scripts
Project root: /Users/phuchdh/Documents/Work/all-in-one-portfolio/backend


In [13]:
class IndicatorConfig:
    """Configuration class for all indicator parameters"""
    
    # Window configurations
    EMA_WINDOWS = [10, 20, 50, 200]
    RSI_WINDOWS = [5, 14]
    ATR_WINDOWS = [10, 14, 252]
    ZSCORE_LOGRETURN_WINDOWS = [10, 20]
    EFI_WINDOWS = [10, 20, 50, 200]
    MFI_WINDOWS = [10, 21]
    RS_WINDOWS = [10, 20, 50, 252]  # 252 for 12-month RS Rating
    ZSCORE_WINDOWS = [20, 50, 200]
    VOLUME_MA_WINDOWS = [10, 20, 50, 200]
    YZ_VOLATILITY_WINDOWS = [10, 20]
    ZSCORE_KF_WINDOWS = [10, 20]
    
    # Other parameters
    VWAP_WINDOW = 200
    DC_THETA = 0.01
    YZ_PERIODS = 252
    
    # Kalman Filter parameters
    KF_TRANSITION_MATRICES = [1]
    KF_OBSERVATION_MATRICES = [1]
    KF_INITIAL_STATE_MEAN = 0
    KF_INITIAL_STATE_COVARIANCE = 1
    KF_OBSERVATION_COVARIANCE = 1
    KF_TRANSITION_COVARIANCE = 0.01

In [22]:
def extract_windowed_data(indicator_result, windows, level_name):
    """Helper function to extract data for different windows and clean column names"""
    result = {}
    for window in windows:
        data = indicator_result.xs(window, level=level_name, axis=1)
        data.columns = data.columns.get_level_values(-1)
        result[window] = data
    return result


def calculate_rs_rating_percentile(rs_data):
    """
    Calculate RS Rating percentile based on relative strength data.
    
    Formula: Percentile = (Number of Stocks Outperformed / Total Number of Stocks) * 100
    RS Rating ranges from 1 to 99, with 99 being the strongest.
    
    Args:
        rs_data (pd.DataFrame): DataFrame with relative strength values for each stock and date
        
    Returns:
        pd.DataFrame: DataFrame with RS Rating percentiles (1-99 scale), NaN preserved
    """
    # For each date (row), calculate percentile rank of each stock
    # pandas.rank() automatically handles NaN values by skipping them in ranking
    rs_rating = rs_data.rank(axis=1, method='min', ascending=True, pct=True) * 100
    
    # Clip to 1-99 range and round, preserving NaN values
    rs_rating = rs_rating.clip(lower=1, upper=99).round()
    
    # Handle NaN values properly - convert to nullable integer type
    # This preserves NaN values while converting valid numbers to integers
    rs_rating = rs_rating.astype('Int64')  # Nullable integer type
    
    # Alternative approaches (uncomment one if needed):
    # Option 1: Fill NaN with 0
    # rs_rating = rs_rating.fillna(0).astype(int)
    
    # Option 2: Fill NaN with 1 (lowest rating)
    # rs_rating = rs_rating.fillna(1).astype(int)
    
    # Option 3: Keep as float to preserve NaN
    # rs_rating = rs_rating.astype(float)  # Keep as float
    
    return rs_rating

def calculate_relative_strength_indicators(stocks: pd.DataFrame, config: IndicatorConfig):
    """Calculate relative strength indicators including RS Rating"""
    rs_indicator = vbt.IndicatorFactory(
        class_name='MansfieldRelativeStrength',
        short_name='mansfield_relative_strength',
        input_names=['close', 'benmark_close'],
        param_names=['window'],
        output_names=['rs', 'mrs']
    ).from_apply_func(relative_strength_nb)
    
    rs_ind = rs_indicator.run(stocks.close, stocks.close.VNINDEX, window=config.RS_WINDOWS)
    
    # Extract and clean columns for rs and mrs
    rs_by_window = {
        w: rs_ind.rs.xs(w, level='mansfield_relative_strength_window', axis=1).rename_axis(None, axis=1)
        for w in config.RS_WINDOWS
    }
    mrs_by_window = {
        w: rs_ind.mrs.xs(w, level='mansfield_relative_strength_window', axis=1).rename_axis(None, axis=1)
        for w in config.RS_WINDOWS
    }
    
    # Rank mrs_by_window for each window
    mrs_rank_by_window = {
        w: df.rank(axis=1, method='min', ascending=False)
        for w, df in mrs_by_window.items()
    }
    
    print(rs_by_window)
    # Calculate RS Rating percentile for all windows
    rs_rating_by_window = {
        w: calculate_rs_rating_percentile(rs_by_window[w])
        for w in config.RS_WINDOWS
    }
    
    return {
        'rs': rs_by_window,
        'mrs': mrs_by_window,
        'mrs_rank': mrs_rank_by_window,
        'rs_rating': rs_rating_by_window
    }


In [ ]:
# Test NaN handling in RS Rating calculation
import numpy as np

# Create sample data with NaN values
test_data = pd.DataFrame({
    'Stock_A': [1.5, 2.0, np.nan, 3.2, 1.8],
    'Stock_B': [2.1, np.nan, 2.8, 2.5, 2.0], 
    'Stock_C': [1.2, 1.8, 2.5, np.nan, 2.2],
    'Stock_D': [3.0, 2.5, 3.5, 3.8, np.nan]
}, index=pd.date_range('2024-01-01', periods=5))

print("Original relative strength data:")
print(test_data)
print("\nRS Rating percentiles (with NaN handling):")

# Test the function
rs_rating_result = calculate_rs_rating_percentile(test_data)
print(rs_rating_result)

print(f"\nData types: {rs_rating_result.dtypes.unique()}")
print(f"Contains NaN: {rs_rating_result.isna().any().any()}")


In [ ]:
import os
import sys
from pathlib import Path
import vectorbt as vbt
from pykalman import KalmanFilter
import pandas as pd
from deltalake import DeltaTable, write_deltalake
from dotenv import load_dotenv
load_dotenv()

# Set up the Python path for notebook environment
# Navigate from backend/scripts to backend (project root)
CURRENT_DIR = os.getcwd()
if 'scripts' in CURRENT_DIR:
    PROJECT_ROOT = os.path.dirname(CURRENT_DIR)  # Go up from scripts to backend
else:
    PROJECT_ROOT = CURRENT_DIR

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Current directory: {CURRENT_DIR}")
print(f"Project root: {PROJECT_ROOT}")

from app.services.indicators import zscore_nb, avwap_func_nb, relative_strength_nb, yang_zhang_volatility_nb, directional_change_nb

Current directory: /Users/phuchdh/Documents/Work/all-in-one-portfolio/backend/scripts
Project root: /Users/phuchdh/Documents/Work/all-in-one-portfolio/backend


## Schema Evolution Strategy for Feature Store

### Overview
Our feature store supports **schema evolution** to handle changes in features over time (like adding RS Rating indicators) without breaking existing data or downstream consumers.

### Schema Evolution Modes

#### 1. **Merge Mode (Recommended for New Features)**
- **Use Case**: Adding new features like `rs_rating_252` 
- **Behavior**: Appends data with new columns, existing data gets NULL for new columns
- **Command**: `sync_features_to_delta_lake(features_data, mode="merge")`

#### 2. **Overwrite Mode (For Major Schema Changes)**
- **Use Case**: Changing data types, removing columns, or complete rebuilds
- **Behavior**: Replaces entire table with new schema
- **Command**: `sync_features_to_delta_lake(features_data, mode="overwrite")`

#### 3. **Append Mode (For Same Schema)**
- **Use Case**: Regular data ingestion with identical schema
- **Behavior**: Adds data without schema changes
- **Command**: `sync_features_to_delta_lake(features_data, mode="append")`

### Schema Versioning

Each write includes metadata for tracking:
- `_feature_store_version`: Schema version (e.g., "1.1" for RS Rating addition)
- `_ingestion_timestamp`: When data was written
- `_schema_hash`: Hash of column names for change detection

### Best Practices

#### ✅ **Safe Schema Changes (Non-Breaking)**
- Adding new features (like RS Rating indicators)
- Adding new technical indicators
- Adding metadata columns

#### ⚠️ **Potentially Breaking Changes**
- Removing existing columns
- Changing data types
- Renaming columns

#### 🚫 **Avoid**
- Removing columns that downstream models depend on
- Changing column semantics without versioning

### Example Evolution Scenarios

#### Scenario 1: Adding RS Rating Features
```python
# Before: ['close', 'volume', 'rsi_5', 'ema_10']
# After:  ['close', 'volume', 'rsi_5', 'ema_10', 'rs_rating_252']
sync_features_to_delta_lake(features_data, mode="merge")  # ✅ Safe
```

#### Scenario 2: Major Refactoring
```python
# Changing all column names or data types
sync_features_to_delta_lake(features_data, mode="overwrite")  # ⚠️ Breaking
```

### Compatibility Validation

The system automatically:
1. **Detects Schema Changes**: Compares old vs new column sets
2. **Reports New Features**: Highlights added RS Rating columns
3. **Warns of Breaking Changes**: Identifies removed columns
4. **Tracks Evolution**: Logs all schema changes with timestamps


In [15]:
def load_stock_data() -> pd.DataFrame:
    """Load stock data from delta lake"""
    try:
        from deltalake import DeltaTable
        storage_options = {
            "AWS_ACCESS_KEY_ID": "JzgMMlm2rZcHlIsV1UBd",
            "AWS_SECRET_ACCESS_KEY": "x872pkjyArcN1LoDjmkqxA4e51xxsJoDyourKaKf",
            "AWS_ENDPOINT_URL": "http://localhost:9000",
            "AWS_ALLOW_HTTP": "true",
            "AWS_EC2_METADATA_DISABLED": "true",
            "AWS_REGION": 'us-east-1',
            "aws_conditional_put": "etag",
        }        
        dt = DeltaTable("s3://delta-table-storage/stocks", storage_options=storage_options)

        # Get data for last 2 years
        now = pd.Timestamp.now()
        start_date = now - pd.DateOffset(years=2)
        df = dt.to_pandas(
            filters=[("date", ">=", start_date)], 
            columns=["symbol", "date", "close", "open", "high", "low", "volume"]
        )
        
        # Convert to the same format as H5 store
        df = df.set_index(["date", "symbol"])
        stocks = df.unstack(level=1).bfill().ffill()
        
        print("Successfully loaded data from delta lake")
        return stocks
        
    except Exception as e:
        print(f"Error loading from delta lake: {e}")
        raise


In [19]:
def run_indicators(stocks: pd.DataFrame) -> pd.DataFrame:
    """Run indicators for building features"""
    config = IndicatorConfig()
    
    print("Calculating relative strength indicators...")
    rs_results = calculate_relative_strength_indicators(stocks, config)
    
    print("Building final features DataFrame...")
    
    # Build simplified features DataFrame focusing on RS Rating
    feature_dict = {
        'close': stocks.close.stack(),
        'volume': stocks.volume.stack(),
        
        # RS Rating (percentile rankings for different windows) - THE NEW FEATURE!
        'rs_rating_10': rs_results['rs_rating'][10].stack(),
        'rs_rating_20': rs_results['rs_rating'][20].stack(),
        'rs_rating_50': rs_results['rs_rating'][50].stack(),
        'rs_rating_252': rs_results['rs_rating'][252].stack(),  # 12-month RS Rating
    }
    
    final_df = pd.DataFrame(feature_dict)
    final_df = final_df.reset_index().rename(columns={'level_1': 'symbol'})
    
    print(f"Generated features DataFrame with shape: {final_df.shape}")
    return final_df


In [23]:
"""Create feature store workflow"""
stock_data = load_stock_data()
if stock_data.empty:
    print("Stock data is empty")

features_data = run_indicators(stock_data)
if features_data.empty:
    print("Features data is empty")

features_data['key'] = features_data['symbol'] + "_" + features_data['date'].astype(str)

Successfully loaded data from delta lake
Calculating relative strength indicators...
{10:                 0001      0530      0533      0570      0573      0577  \
date                                                                     
2023-09-05       NaN       NaN       NaN       NaN       NaN       NaN   
2023-09-06       NaN       NaN       NaN       NaN       NaN       NaN   
2023-09-07       NaN       NaN       NaN       NaN       NaN       NaN   
2023-09-08       NaN       NaN       NaN       NaN       NaN       NaN   
2023-09-11       NaN       NaN       NaN       NaN       NaN       NaN   
...              ...       ...       ...       ...       ...       ...   
2025-08-25 -1.063800 -1.063800 -1.063800 -1.063800 -1.063800 -1.063800   
2025-08-26 -3.562543 -3.562543 -3.562543 -3.562543 -3.562543 -3.562543   
2025-08-27 -3.657388 -3.657388 -3.657388 -3.657388 -3.657388 -3.657388   
2025-08-28 -2.389851 -2.389851 -2.389851 -2.389851 -2.389851 -2.389851   
2025-08-29 -3.103653 -

,date,symbol,close,volume,rs_rating_10,rs_rating_20,rs_rating_50,rs_rating_252
0,2023-09-05,0001,117.144806,12343400.0,<NA>,<NA>,<NA>,<NA>
1,2023-09-05,0530,110.654778,2657000.0,<NA>,<NA>,<NA>,<NA>
2,2023-09-05,0533,119.219177,2657000.0,<NA>,<NA>,<NA>,<NA>
3,2023-09-05,0570,171.870987,9552200.0,<NA>,<NA>,<NA>,<NA>
4,2023-09-05,0573,207.809204,9440300.0,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...
178418,2025-08-29,VPI,57.000000,2095600.0,83,80,68,30
178419,2025-08-29,VRE,30.400000,9892100.0,81,79,82,91
178420,2025-08-29,VSC,30.000000,11183500.0,1,90,99,99
178421,2025-08-29,VTP,99.000000,557300.0,3,3,1,84


In [42]:
test_df = features_data[features_data['symbol'] == 'VHM']
test_df.tail(10)

,date,symbol,close,volume,rs_rating_10,rs_rating_20,rs_rating_50,rs_rating_252
175155,2025-08-18,VHM,93.900002,4501300.0,65,59,87,99
175514,2025-08-19,VHM,93.900002,4158500.0,6,9,80,99
175873,2025-08-20,VHM,99.000000,9090600.0,71,75,88,99
176232,2025-08-21,VHM,99.800003,4878600.0,84,79,92,99
176591,2025-08-22,VHM,98.099998,6239700.0,87,82,93,99
176950,2025-08-25,VHM,98.400002,5063900.0,94,87,96,99
177309,2025-08-26,VHM,105.199997,6413000.0,96,88,96,99
177668,2025-08-27,VHM,104.599998,3023700.0,97,90,94,99
178027,2025-08-28,VHM,104.599998,3292800.0,97,91,94,99
178386,2025-08-29,VHM,104.500000,4250800.0,97,90,94,99
